# Generator notebook

This notebook is used to prototype the generator python application.

The cell structure is intended to replicate the general structure of the generator script.

1. Import libraries and set basic variables
2. Load the configuration
3. Load the base demand
4. Execute transformations
5. Write output

In [1]:
# 1. Import libraries and set basic variables

import numpy as np
import pandas as pd
from itertools import product
import pyarrow as pa
import pyarrow.dataset as ds
import pyarrow.compute as pc
from pathlib import Path
import yaml
import duckdb
from datetime import datetime, timedelta
import time

import sys
sys.path.append(str(Path(globals()['_dh'][0]).resolve().parent.parent))

from generator.paths import project_path, pipeline_path, input_path, data_path, output_path
from generator.library.utilities import sort_params, clear_dir, ktok
from generator.library.loaders import base_loader_csv
from generator.transformers.dimension import segment_constant_factor_split, segment_add_total, segment_append_constant, segment_append_percentage, segment_append_reminder, segment_remove
from generator.transformers.scenario import create_base_scenario
from generator.library.db import read_data, delete_duckdb_file
import generator.library.scenario_constraints
from generator.library.scenario import process_scenario, process_scenario_sql


In [2]:
# 2. Load configuration (new version, come back to this later)

with open(project_path / 'config.yaml', "r") as f:
    config = yaml.safe_load(f)

with open(pipeline_path / 'county-prototype.yaml', "r") as f:
    pipeline = yaml.safe_load(f)

## Set flag to clear the api folder
clear_api_flag = True

## Set common variables

data_path = output_path
db_file = data_path / "core.duckdb"
default_table = "demand"

full_partition = ['scenario_id', 'geography', 'segment', 'timestamp_year']

base_schema = ['geography', 'segment', 'timestamp', 'value']
base_schema_map = {
    'geography': "geography",
    'segment': "segment", 
    'timestamp': "timestamp", 
    'value': "value"
}

base_scenario = "default"
include_base_scenario = True

scenario_schema = [
    'housing_electrification',
    'transport_electrification',
    'industry_transition'
    ]

scenario_params = {
    "housing_electrification": {
        "type": "curve",
        "geography": "all",
        "segment": "housing",
        "how": "multiply",
        "curve_path_template": str(
            input_path
            / "scenario_housing_electrification"
            / "scenario_housing_electrification,i={value},start_year=2025,end_year=2050.parquet"
        ),
        "parameters": [0, 1, 2, 3, 4],
        "default": 2,
    },
    "transport_electrification": {
        "type": "curve",
        "geography": "all",
        "segment": "transport",  # was 'industry'; corrected
        "how": "multiply",
        "curve_path_template": str(
            input_path
            / "scenario_transport_electrification"
            / "scenario_transport_electrification,i={value},start_year=2025,end_year=2050.parquet"
        ),
        "parameters": [0, 1, 2, 3, 4],
        "default": 2,
    },
    "industry_transition": {
        "type": "curve",
        "geography": "all",
        "segment": "industry",  # was 'housing'; corrected
        "how": "multiply",
        "curve_path_template": str(
            input_path
            / "scenario_industry_transition"
            / "scenario_industry_transition,i={value},start_year=2025,end_year=2050.parquet"
        ),
        "parameters": [0, 1, 2],
        "default": 1,
    },
}

full_schema = scenario_schema + base_schema

'''
# This is the full set of scenarios

scenario_schema = [
    'housing_electrification',
    'transport_electrification',
    'industry_transition',
    'population',
    'new_industry',
    'new_datacenters',
    'flexibility',
    'energy_efficiency'
    ]
'''

base_data = "base-demand/base-load-curve,aggregation=mean,base-year=2024,geography=00,normalized=False,resolution=1h.csv"

In [3]:
# Clear the output path to prepare for new data

clear_dir(output_path)

In [4]:
# 3. Calculate the scenarios

# Extract names and value lists
names = list(scenario_params.keys())
values = [scenario_params[name]['parameters'] for name in names]

# Define default scenario
default_scenario = {
    name: scenario_params[name]["default"]
    for name in names
}

# Generate all combinations
all_scenarios = [dict(zip(names, combo)) for combo in product(*values)]

# Mark the default is base is not kept (and is default)
if include_base_scenario:
    scenarios = all_scenarios
else:
    scenarios = []
    for s in all_scenarios:
        s_out = dict(s)
        if s == default_scenario:
            s_out["default"] = True
        scenarios.append(s_out)

In [5]:
# 3. Load the base demand

base_loader_csv(base_schema, base_schema_map, input_path / base_data, db_file)

{'target': '/home/viktor/code/behovskartan/generator/output/core.duckdb',
 'table': 'demand',
 'added_rows': 8760,
 'added_columns': ['geography', 'segment', 'timestamp', 'value']}

In [6]:
# 4 Segment the data (temporary)

# 4.1 Split the data by geographies
geo_file = input_path / "county_energy_split/county_energy_split.csv"
geographies = pd.read_csv(geo_file, dtype={"geography": str, "factor": float})

segment_constant_factor_split(db_file, geographies)

# 4.2 Split the data by segments
segment_append_constant(db_file, 'segment', 'industry', 100)
segment_append_percentage(db_file, 'segment', 'total', 'transport', 0.15)
segment_append_reminder(db_file, 'segment', 'total', ['industry', 'transport'], 'housing')
segment_remove(db_file, 'segment', 'total')

{'target': '/home/viktor/code/behovskartan/generator/output/core.duckdb',
 'table': 'demand',
 'rows': -183960,
 'columns': []}

In [7]:
create_base_scenario(
    in_data=db_file, 
    base_year=2024,
    start_year=2025,
    end_year=2050,
    out_data=data_path, 
    partition=full_partition,
    growth_curve=input_path / "scenario_base/scenario_base_growth,start_year=2025,end_year=2050.parquet",
    base_scenario=base_scenario
)

{'target': '/home/viktor/code/behovskartan/generator/output',
 'status': 'done',
 'rows_written': 14357952,
 'years': {'start': 2025, 'end': 2050},
 'scenario_id': 'default'}

In [8]:
delete_duckdb_file(db_file)

True

In [ ]:
# Test with SQL version
scenario_ids = []
for scn in scenarios:
    start_time = time.perf_counter()
    sid = process_scenario_sql(
        scenario=scn,
        scenario_params=scenario_params,
        scenario_schema=scenario_schema,
        output_path=output_path,
        partition=full_partition,
        base_scenario=base_scenario,
    )
    scenario_ids.append(sid)
    print(f"Finished in {time.perf_counter() - start_time:.2f} s")

print("Scenarios written (SQL):", scenario_ids)


[SQL] Processing scenario: housing_electrification=0,transport_electrification=0,industry_transition=0
Finished in 16.31 s
[SQL] Processing scenario: housing_electrification=0,transport_electrification=0,industry_transition=1
Finished in 17.70 s
[SQL] Processing scenario: housing_electrification=0,transport_electrification=0,industry_transition=2
Finished in 21.64 s
[SQL] Processing scenario: housing_electrification=0,transport_electrification=1,industry_transition=0
Finished in 35.58 s
[SQL] Processing scenario: housing_electrification=0,transport_electrification=1,industry_transition=1
Finished in 22.93 s
[SQL] Processing scenario: housing_electrification=0,transport_electrification=1,industry_transition=2
Finished in 21.05 s
[SQL] Processing scenario: housing_electrification=0,transport_electrification=2,industry_transition=0
Finished in 21.35 s
[SQL] Processing scenario: housing_electrification=0,transport_electrification=2,industry_transition=1
Finished in 36.32 s
[SQL] Processing

**The code below all pertains to writing data.**

In [ ]:
## Clear the api folder
if clear_api_flag:
    print("Clearing API folder...")
    clear_dir(data_path)

Clearing API folder...


In [ ]:
# Partition dataframe and write to parquet with pyarrow

use_scenario_id = config['generator']['useScenarioId']
partition_keys = config['generator']['partitionKeys']
row_group_size = config['generator']['rowGroupSize']

# 1) If scenario._id is used, compute and append it directly in the DataFrame
if use_scenario_id:
    print("Adding scenario._id to DataFrame...")

    scenario_cols = sorted(
        col for col in extended_geography_scenario_sector_demand.columns
        if col.startswith('scenario.')
    )

    extended_geography_scenario_sector_demand['scenario._id'] = (
        extended_geography_scenario_sector_demand[scenario_cols]
        .astype(str)
        .rename(columns=lambda col: col.split('.', 1)[1])  # remove "scenario." prefix
        .agg(lambda row: '+'.join(f"{k}:{v}" for k, v in row.items()), axis=1)
    )

# 2) Convert your pandas DF into an Arrow Table
print("Converting DataFrame to Arrow Table...")
table = pa.Table.from_pandas(
    extended_geography_scenario_sector_demand,
    preserve_index=False
)

actual_partitions = []

# 3) Compute derived keys and append
print("Computing derived partition keys...")
for entry in partition_keys:
    if isinstance(entry, dict) and "transform" in entry:
        src = entry["path"]
        tf  = entry["transform"]
        arr = getattr(pc, tf)(table[src])
        table = table.append_column(tf, arr)
        actual_partitions.append(tf)
    elif isinstance(entry, dict):
        actual_partitions.append(entry['path'])

# 4) Write Parquet dataset partitioned on actual columns
print("Writing Parquet dataset...")
ds.write_dataset(
    data=table,
    base_dir=str(data_path),
    format="parquet",
    partitioning=actual_partitions,
    partitioning_flavor="hive",
    file_options=ds.ParquetFileFormat()
                   .make_write_options(compression="snappy"),
    min_rows_per_group=row_group_size,
    max_rows_per_group=row_group_size,
    existing_data_behavior="overwrite_or_ignore"
)


Adding scenario._id to DataFrame...
Converting DataFrame to Arrow Table...
Computing derived partition keys...
Writing Parquet dataset...
